### Дома по регионам

In [ ]:
import pandas as pd
import psycopg2
from ydata_profiling import ProfileReport
import os
import re
import logging
import json

# Настройка логирования
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logging.info("Начало выполнения скрипта")

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Заменить на актуальный пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос
query = """
SELECT number_of_rooms, total_area, price, currency, regionname, cityname, createdtime,
       isactive, total_floors, furnished, repairs, comission, wc, house_type,
       total_living_area, water, heating, gas, electricity, location, plot
FROM houses
WHERE total_area IS NOT NULL AND total_area >= 18
    AND price IS NOT NULL AND price > 0
    AND createdtime >= '2024-02-01';
"""

# Загружаем данные из PostgreSQL
df = pd.read_sql(query, conn)
conn.close()  # Закрываем соединение

# Проверка данных перед конвертацией
df = df[(df["total_area"] > 0) & (df["price"] > 0)]

# Актуальный курс валют
usd_to_uzs = 12600

# Функция конвертации цен
def convert_prices(row):
    if row["currency"] == "UYE":
        price_uye = row["price"]
        price_uz = price_uye * usd_to_uzs
    else:  # Если валюта в UZS
        price_uz = row["price"]
        price_uye = price_uz / usd_to_uzs

    # Проверяем, что цена не за 1 м², а за всю квартиру
    if price_uye > 5000:  # Если стоимость квартиры в UYE больше 5000 → это вся квартира
        price_per_m2_uye = price_uye / row["total_area"]
    else:  # Иначе это цена за 1 м²
        price_per_m2_uye = price_uye
        price_uye *= row["total_area"]

    if price_uz > 65003698:  # Аналогично для UZS
        price_per_m2_uz = price_uz / row["total_area"]
    else:
        price_per_m2_uz = price_uz
        price_uz *= row["total_area"]

    return pd.Series([price_uye, price_uz, price_per_m2_uye, price_per_m2_uz])

# Применяем конвертацию
df[["price_uye", "price_uz", "price_per_m2_uye", "price_per_m2_uz"]] = df.apply(convert_prices, axis=1)

# Удаляем ненужную колонку "currency"
if "currency" in df.columns:
    df.drop(columns=["currency"], inplace=True)

# Функция для распарсивания JSON-колонки
def parse_json_column(df, column_name):
    try:
        # Преобразуем JSON-строки в словари
        df[column_name] = df[column_name].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

        # Проверяем, есть ли вложенные данные в колонке
        if df[column_name].apply(lambda x: isinstance(x, dict)).any():
            parsed_df = df[column_name].apply(pd.Series).add_prefix(f"{column_name}_")
            df = pd.concat([df, parsed_df], axis=1)

        df.drop(columns=[column_name], inplace=True)  # Удаляем оригинальную колонку
    except Exception as e:
        logging.warning(f"Ошибка при обработке колонки {column_name}: {e}")
    return df

# Применяем функцию к каждой JSON-колонке
json_columns = ["water", "heating", "gas", "electricity", "location", "plot"]
for column in json_columns:
    if column in df.columns:
        df = parse_json_column(df, column)


# Фильтруем регионы с более чем 500 публикациями
region_counts = df["regionname"].value_counts()
valid_regions = region_counts[region_counts > 100].index

# Создаем папку для отчетов
output_dir = "houses_reports"
os.makedirs(output_dir, exist_ok=True)

# Функция для очистки имен файлов
def clean_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", name)

# Генерируем отчеты ydata_profiling по каждому региону
for region in valid_regions:
    region_df = df[df["regionname"] == region]
    if not region_df.empty:  # Проверка на пустоту
        profile = ProfileReport(region_df, title=f"Отчет по региону {region}", explorative=True)
        profile.to_file(f"{output_dir}/profile_{clean_filename(region)}.html")
        logging.info(f"Отчет для региона {region} сохранен.")

logging.info("Готово! Отчеты сохранены в файлы.")